# 02 · Silver worker
`bronze.per_all_people_f` → `silver.hcmworker` (SCD2, module 06 §1).

✅ Expect **2,693** rows · 20 `IsDobMissing` · EMP 2,566 / CWK 127.

In [0]:
from pyspark.sql import functions as F, Window
from hcm_utils import load_config, parse_date, latest_batch, issue_rows, write_issues, scd2_merge
#confog read
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("run_id", "manual")
catalog = dbutils.widgets.get("catalog")
run_id  = dbutils.widgets.get("run_id")
cfg = load_config()

In [0]:
SRC_FILE = "PER_ALL_PEOPLE_F"
COLS = ["PERSON_ID", "PERSON_NUMBER", "FULL_NAME", "DATE_OF_BIRTH", "SEX", "PERSON_TYPE", "NATIONAL_IDENTIFIER"]

raw = latest_batch(f"{catalog}.bronze.per_all_people_f").select(*COLS)

issues = [issue_rows(raw.filter("DATE_OF_BIRTH IS NULL"), SRC_FILE, "PERSON_ID",
                     "NULL_VALUE", "DATE_OF_BIRTH is null", "FLAGGED")]

# exact duplicates
df = raw.withColumn("_rn", F.row_number().over(Window.partitionBy(*COLS).orderBy(F.lit(1))))
issues.append(issue_rows(df.filter("_rn > 1"), SRC_FILE, "PERSON_ID",
                         "DUPLICATE_ROW", "Exact re-send duplicate of an existing PERSON_ID", "DROPPED"))
df = df.filter("_rn = 1").drop("_rn")

assert df.groupBy("PERSON_ID").count().filter("count > 1").count() == 0, "PERSON_ID not unique after dedup"

In [0]:
silver = df.select(
    F.col("PERSON_ID").cast("bigint").alias("PersonId"),
    F.trim("PERSON_NUMBER").alias("PersonNumber"),
    F.trim("FULL_NAME").alias("FullName"),
    parse_date("DATE_OF_BIRTH").alias("DateOfBirth"),
    F.upper(F.trim("SEX")).alias("Gender"),
    F.upper(F.trim("PERSON_TYPE")).alias("PersonType"),
    F.trim("NATIONAL_IDENTIFIER").alias("NationalIdentifier"),
    F.col("DATE_OF_BIRTH").isNull().alias("IsDobMissing"),
    F.lit(run_id).alias("RunId"),
)

TRACKED = ["PersonNumber", "FullName", "DateOfBirth", "Gender", "PersonType", "NationalIdentifier"]
scd2_merge(silver, f"{catalog}.silver.hcmworker", key="PersonId", tracked_cols=TRACKED)
print("issues logged:", write_issues(issues, catalog, run_id))
print("hcmworker current rows:", spark.table(f"{catalog}.silver.hcmworker").filter("IsCurrent").count())